# A1：Multi-Agent 死鎖實作與修復 — LangGraph 實戰

> **對應 Blog：** FDE 面試準備指南（十六）Multi-Agent 狀態管理與死鎖排除  
> **核心目標：** 親手跑出死鎖，再親手修好它。讀文章永遠比不上踩過這個坑。

## 學習路徑

| 步驟 | 內容 | 目的 |
|------|------|------|
| Part 0 | 環境設定 | 安裝依賴 |
| Part 1 | **製造死鎖** | 理解為什麼會發生 |
| Part 2 | State Reducer 設計 | 防止 Race Condition |
| Part 3 | 加入迭代護欄 | 確保圖一定收斂 |
| Part 4 | Checkpoint 持久化 | 模擬 Worker 崩潰恢復 |
| Part 5 | **面試白板語言** | 從 code 走回口語表達 |

---

## 面試情境（你要能回答這題）

> 「客戶使用 LangGraph 部署了一個階層式的 Multi-Agent 系統。Router Agent 分發任務給法務審查 Agent 和財務計算 Agent。上線後，特定的複雜查詢會導致系統 Timeout，或是多個 Agent 互相死循環呼叫。你在 Google Doc 看到對話日誌，如何定位問題？架構上如何設計 State Management 與護欄？」

## Part 0：環境設定

In [ ]:
# !pip install langgraph langchain-openai python-dotenv
import os
from dotenv import load_dotenv
load_dotenv()

# 如果沒有 .env，直接設定
# os.environ["OPENAI_API_KEY"] = "sk-..."

print("API Key 已設定：", "OPENAI_API_KEY" in os.environ)

In [ ]:
import time
import json
from typing import TypedDict, Annotated, Literal
import operator

from langgraph.graph import StateGraph, END, START
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

# 使用便宜的模型做測試
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("LangGraph + OpenAI 初始化完成")

## Part 1：製造死鎖（壞的設計）

先故意寫一個**有問題的架構**，讓死鎖真的發生。

**問題根源：**
- 法務 Agent 和財務 Agent 的職責邊界模糊
- 兩者都覺得需要對方確認才能完成
- 全域 State 互相覆寫
- 沒有終止條件

In [ ]:
# ❌ 有問題的 State 設計（多個 Agent 共用同一個欄位）
class BadState(TypedDict):
    messages: list[str]
    current_agent: str
    result: str          # ← 危險：兩個 Agent 都會寫這個
    iteration: int
    task: str

In [ ]:
# ❌ 壞的 Agent 設計：每個 Agent 都需要對方才能完成

def bad_legal_agent(state: BadState) -> BadState:
    """法務 Agent — 但它會無限請求財務確認"""
    state["messages"].append(f"[法務 Agent] 分析合約中...（第 {state['iteration']} 次）")
    print(f"[{state['iteration']}] 法務 Agent 執行")
    
    # 模擬：法務 Agent 覺得需要財務確認才能完成
    state["result"] = "pending_finance_confirm"
    state["current_agent"] = "finance"
    state["iteration"] += 1
    return state

def bad_finance_agent(state: BadState) -> BadState:
    """財務 Agent — 但它也需要法務確認"""
    state["messages"].append(f"[財務 Agent] 計算財務影響中...（第 {state['iteration']} 次）")
    print(f"[{state['iteration']}] 財務 Agent 執行")
    
    # 模擬：財務 Agent 覺得需要法務確認才能完成
    state["result"] = "pending_legal_confirm"
    state["current_agent"] = "legal"
    state["iteration"] += 1
    return state

def bad_router(state: BadState) -> Literal["legal", "finance", "end"]:
    """Router 沒有終止條件 ← 這就是死鎖的根源"""
    if state["current_agent"] == "legal":
        return "legal"
    elif state["current_agent"] == "finance":
        return "finance"
    return "end"

print("壞的 Agent 設計完成")

In [ ]:
# ❌ 建立有死鎖風險的 Graph

bad_graph_builder = StateGraph(BadState)

bad_graph_builder.add_node("legal", bad_legal_agent)
bad_graph_builder.add_node("finance", bad_finance_agent)

bad_graph_builder.add_edge(START, "legal")
bad_graph_builder.add_conditional_edges("legal", bad_router)
bad_graph_builder.add_conditional_edges("finance", bad_router)

bad_graph = bad_graph_builder.compile()

print("死鎖圖建立完成 — 準備觸發死鎖")

In [ ]:
# 🔥 觸發死鎖（用計數器防止真的無限運行）
# 在真實系統中，這會一直跑到 Timeout

MAX_DEMO_ITER = 8  # 只跑 8 次讓大家看清楚模式

initial_bad_state: BadState = {
    "messages": [],
    "current_agent": "legal",
    "result": "",
    "iteration": 1,
    "task": "審閱採購合約，金額 500 萬台幣"
}

print("=" * 50)
print("⚠️  死鎖模擬開始（限制 8 次迭代）")
print("=" * 50)

# 用 stream 逐步觀察
step_count = 0
for step in bad_graph.stream(initial_bad_state):
    step_count += 1
    node_name = list(step.keys())[0]
    state_snapshot = step[node_name]
    print(f"步驟 {step_count}: [{node_name}] → result='{state_snapshot.get('result', '')}', iteration={state_snapshot.get('iteration', 0)}")
    
    if step_count >= MAX_DEMO_ITER:
        print("\n💥 強制停止！在真實系統中，這裡會 TIMEOUT")
        break

print("\n📋 死鎖日誌（你在 Google Doc 看到的）:")
for msg in state_snapshot.get("messages", []):
    print(" ", msg)

### 死鎖診斷：你從日誌看到什麼？

```
診斷步驟：

Step 1：同一個 Agent 被呼叫次數一直增加 → 確認是無限迴圈
Step 2：呼叫鏈是 法務 → 財務 → 法務 → 財務 → 雙向等待（典型循環依賴）
Step 3：result 一直在 pending_xxx → State 沒有推進
Step 4：根本原因：
  ├── 兩個 Agent 的任務邊界沒有清晰定義
  ├── 缺乏「我已完成」的退出信號
  └── 沒有 max_iteration 護欄
```

## Part 2：修復 — 正確的 State 設計（State Reducer）

**三個修復方向：**
1. 每個 Agent 只寫自己的 substate（隔離）
2. Append-only messages（不覆寫）
3. 每個 Agent 能獨立產出確定性輸出（不依賴對方）

In [ ]:
# ✅ 正確的 State 設計

class GoodState(TypedDict):
    task: str
    # Append-only：用 operator.add 作為 reducer
    messages: Annotated[list[str], operator.add]
    
    # 每個 Agent 只能寫自己的欄位（substate 隔離）
    legal_output: dict | None    # 只有 legal_agent 寫
    finance_output: dict | None  # 只有 finance_agent 寫
    final_decision: str | None   # 只有 router 寫
    
    # 護欄計數器（全域可見）
    iteration_count: int

print("✅ 正確的 State 設計：")
print("  - messages: Append-only（用 operator.add）")
print("  - legal_output: 只有 legal agent 能寫")
print("  - finance_output: 只有 finance agent 能寫")
print("  - iteration_count: 護欄計數器")

In [ ]:
# ✅ 正確的 Agent 設計：每個 Agent 能獨立完成自己的子任務

def legal_agent(state: GoodState) -> dict:
    """法務 Agent — 獨立分析，輸出確定性結果（不依賴財務 Agent）"""
    print(f"[法務 Agent] 執行中（iteration={state['iteration_count']}）")
    
    # 模擬 LLM 分析（真實場景呼叫 Gemini/OpenAI）
    legal_result = {
        "risk_level": "MEDIUM",
        "risk_reasons": [
            "合約缺乏違約金上限條款",
            "付款條件偏向供應商"
        ],
        "recommendation": "建議修改第 3.2 條款後再簽署"
    }
    
    # 只寫自己的 substate，不碰 finance_output
    return {
        "legal_output": legal_result,           # ← 確定性輸出
        "messages": [f"[法務] 風險評級: {legal_result['risk_level']}"],  # ← append
        "iteration_count": state["iteration_count"] + 1
    }

def finance_agent(state: GoodState) -> dict:
    """財務 Agent — 獨立計算財務影響（可選讀取法務結果，但不等待法務確認）"""
    print(f"[財務 Agent] 執行中（iteration={state['iteration_count']}）")
    
    # 可以讀取法務結果（如果已有的話）做為參考，但不依賴它
    legal_risk = state.get("legal_output", {}).get("risk_level", "UNKNOWN")
    
    finance_result = {
        "contract_value": 5_000_000,
        "currency": "TWD",
        "payment_risk": "LOW",
        "note": f"法務風險評級參考: {legal_risk}"
    }
    
    # 只寫自己的 substate
    return {
        "finance_output": finance_result,        # ← 確定性輸出
        "messages": [f"[財務] 合約金額: {finance_result['contract_value']:,} TWD"],
        "iteration_count": state["iteration_count"] + 1
    }

print("✅ 正確的 Agent 設計完成")

In [ ]:
# ✅ Router 和 Fallback 設計

MAX_ITERATIONS = 5  # 護欄上限

def router_agent(state: GoodState) -> dict:
    """Router：收集兩者輸出，綜合決策"""
    print(f"[Router] 綜合判斷中")
    
    legal = state.get("legal_output", {})
    finance = state.get("finance_output", {})
    
    # 只有 Router 做最終決策
    if legal.get("risk_level") == "HIGH":
        decision = "REJECT — 法律風險過高"
    elif finance.get("payment_risk") == "HIGH":
        decision = "REJECT — 財務風險過高"
    else:
        decision = f"APPROVE with conditions: {legal.get('recommendation', '')}"
    
    return {
        "final_decision": decision,
        "messages": [f"[Router] 最終決策: {decision}"]
    }

def fallback_node(state: GoodState) -> dict:
    """Fallback：超過迭代次數上限時的兜底"""
    print(f"[Fallback] 觸發人工審核")
    return {
        "final_decision": "ESCALATE — 系統無法自動完成，已轉交人工審核",
        "messages": [f"[Fallback] 超過 {MAX_ITERATIONS} 次迭代，觸發人工審核"]
    }

# 條件路由：每條邊都先檢查護欄
def should_proceed_to_finance(state: GoodState) -> Literal["finance", "fallback"]:
    if state["iteration_count"] >= MAX_ITERATIONS:
        return "fallback"
    return "finance"

def should_proceed_to_router(state: GoodState) -> Literal["router", "fallback"]:
    if state["iteration_count"] >= MAX_ITERATIONS:
        return "fallback"
    # 兩個 Agent 都完成才進 Router
    if state.get("legal_output") and state.get("finance_output"):
        return "router"
    return "fallback"  # 異常情況兜底

print("✅ Router + Fallback + 護欄設計完成")

## Part 3：建立收斂的 Graph（帶護欄）

In [ ]:
# ✅ 建立正確的 Graph

graph_builder = StateGraph(GoodState)

# 加入所有節點
graph_builder.add_node("legal", legal_agent)
graph_builder.add_node("finance", finance_agent)
graph_builder.add_node("router", router_agent)
graph_builder.add_node("fallback", fallback_node)

# 邊的設計：
# START → legal（法務先跑）
# legal → finance（帶護欄的條件邊）
# finance → router 或 fallback（帶護欄）
# router → END
# fallback → END

graph_builder.add_edge(START, "legal")
graph_builder.add_conditional_edges("legal", should_proceed_to_finance)
graph_builder.add_conditional_edges("finance", should_proceed_to_router)
graph_builder.add_edge("router", END)
graph_builder.add_edge("fallback", END)

good_graph = graph_builder.compile()

print("✅ 收斂 Graph 建立完成")
print("\nGraph 結構：")
print("  START → legal")
print("  legal → finance（如果 iteration < 5）")
print("  legal → fallback（如果 iteration >= 5）")
print("  finance → router（如果兩個 output 都有）")
print("  finance → fallback（異常情況）")
print("  router → END")
print("  fallback → END")

In [ ]:
# 執行正確的 Graph

initial_good_state: GoodState = {
    "task": "審閱採購合約，金額 500 萬台幣",
    "messages": [],
    "legal_output": None,
    "finance_output": None,
    "final_decision": None,
    "iteration_count": 0
}

print("=" * 50)
print("✅ 正確的 Multi-Agent 執行")
print("=" * 50)

final_state = good_graph.invoke(initial_good_state)

print("\n📋 執行日誌：")
for msg in final_state["messages"]:
    print(" ", msg)

print("\n✅ 最終決策：", final_state["final_decision"])
print("✅ 總迭代次數：", final_state["iteration_count"])
print("✅ 法務輸出：", final_state["legal_output"])
print("✅ 財務輸出：", final_state["finance_output"])

## Part 4：模擬 Checkpoint — Worker 崩潰後從斷點恢復

在真實環境中，Checkpoint 存到 Redis（熱狀態）和 Firestore（持久化）。  
這裡用 dict 模擬，讓你理解設計原理。

In [ ]:
import copy

# 模擬 Redis/Firestore 的 Checkpoint 存儲
checkpoint_store: dict = {}

def save_checkpoint(task_id: str, step: str, state: dict):
    """模擬寫入 Firestore"""
    checkpoint_store[task_id] = {
        "step": step,
        "state": copy.deepcopy(state),
        "timestamp": time.time()
    }
    print(f"💾 [Checkpoint] 儲存步驟 '{step}'，task_id={task_id}")

def load_checkpoint(task_id: str) -> dict | None:
    """模擬從 Firestore 讀取最後 Checkpoint"""
    return checkpoint_store.get(task_id)

# 模擬帶 Checkpoint 的 Agent 執行流程
def run_with_checkpoint(task_id: str, initial_state: GoodState):
    # 檢查是否有 Checkpoint（斷點恢復）
    cp = load_checkpoint(task_id)
    if cp:
        print(f"🔄 [恢復] 從 Checkpoint '{cp['step']}' 繼續執行")
        state = cp["state"]
        # 根據上次到哪個步驟，決定從哪裡繼續
        resume_from = cp["step"]
    else:
        print(f"🆕 [新任務] 從頭開始，task_id={task_id}")
        state = initial_state
        resume_from = "legal"
    
    # Step 1: Legal Agent（如果還沒跑過）
    if resume_from == "legal":
        result = legal_agent(state)
        state.update(result)
        state["messages"] = state.get("messages", []) + result.get("messages", [])
        save_checkpoint(task_id, "finance", state)  # 法務完成後存 CP
        resume_from = "finance"
    
    # Step 2: Finance Agent（如果還沒跑過）
    if resume_from == "finance":
        result = finance_agent(state)
        state.update(result)
        state["messages"] = state.get("messages", []) + result.get("messages", [])
        save_checkpoint(task_id, "router", state)  # 財務完成後存 CP
        resume_from = "router"
    
    # Step 3: Router
    if resume_from == "router":
        result = router_agent(state)
        state.update(result)
        state["messages"] = state.get("messages", []) + result.get("messages", [])
    
    return state


# === 情境 1：正常第一次執行 ===
print("=" * 50)
print("情境 1：正常執行（假設 Worker 在法務完成後崩潰）")
print("=" * 50)

init = {
    "task": "審閱採購合約，金額 500 萬台幣",
    "messages": [],
    "legal_output": None,
    "finance_output": None,
    "final_decision": None,
    "iteration_count": 0
}

# 模擬：只執行法務就「崩潰」
result = legal_agent(init)
init.update(result)
save_checkpoint("task-001", "finance", init)  # 法務完成後儲存
print("💥 Worker 崩潰！")

print()

# === 情境 2：Worker 重啟，從 Checkpoint 恢復 ===
print("=" * 50)
print("情境 2：新 Worker 從 Checkpoint 恢復")
print("=" * 50)

fresh_init = {
    "task": "審閱採購合約，金額 500 萬台幣",
    "messages": [],
    "legal_output": None,
    "finance_output": None,
    "final_decision": None,
    "iteration_count": 0
}

final = run_with_checkpoint("task-001", fresh_init)
print("\n✅ 最終結果：", final["final_decision"])
print("✅ 法務 Agent 沒有重跑（節省 Token 成本）")

## Part 5：從 Code 走回白板語言

這是最重要的 Part。實作完之後，你要能用面試語言說出來。

---

### 面試答題框架（對著鏡子練）

**問題有兩個層面：**

**第一層：架構設計問題**  
法務 Agent 和財務 Agent 的職責邊界沒有明確定義，導致兩者互相等待，形成循環依賴。根本解法是讓每個 Agent 能獨立輸出確定性結果，由 Router 負責綜合判斷，而不是讓子 Agent 互相協調。

**第二層：State Management 問題**  
多個 Agent 同時寫入 Global State 會有 Race Condition。解法是用 Append-only Reducer，每個 Agent 只能寫入自己的 substate，確保互相隔離。

**護欄設計：**  
在每條 Conditional Edge 上加 `iteration_count >= 5` 的硬性跳出，超過就轉 Fallback，觸發 Alert，等待人工處理。

**狀態持久化：**  
用 Redis 存 Hot Path 的 Checkpoint，用 Firestore 存完整任務歷史，確保 Worker 崩潰後能從斷點續傳，不重複消耗 Token 成本。

In [ ]:
# Trade-off 速查表

tradeoffs = {
    "MAX_ITERATIONS 設多少？": {
        "2-3": "成本低，但複雜任務可能被過早 Fallback",
        "5（推薦）": "大多數正常任務在 5 次內完成，超過通常是設計問題",
        "10+": "Token 成本高，且問題可能早就不在迭代次數"
    },
    "State 存哪裡？": {
        "Redis（推薦 Hot Path）": "< 1ms 讀寫，但重啟後資料可能消失",
        "Firestore（推薦持久化）": "10~50ms，強持久，支援查詢（合規需求）",
        "記憶體（只適合 demo）": "最快，但 Worker 崩潰就全沒了"
    },
    "Agent 能並行嗎？": {
        "法務和財務可以並行": "如果法務輸出不影響財務的輸入（本案可以）",
        "順序執行": "如果財務需要法務的風險評級才能計算（有依賴）",
        "關鍵判斷": "先畫出 Agent 間的 data flow，再決定並行還是順序"
    }
}

for question, options in tradeoffs.items():
    print(f"\n❓ {question}")
    for opt, explanation in options.items():
        print(f"   {opt}: {explanation}")

## 總結：你踩過的坑

| 坑 | 原因 | 修復 |
|---|---|---|
| 死鎖無限迴圈 | Agent 職責邊界模糊，互相等待 | 每個 Agent 獨立輸出確定性結果 |
| Race Condition | 多 Agent 覆寫同一 State 欄位 | Append-only + substate 隔離 |
| 無法收斂 | 沒有終止條件 | `iteration_count >= MAX` → Fallback |
| Worker 崩潰全部重跑 | 沒有 Checkpoint | 每個步驟完成後儲存 State |

**下一步：** 參考 `A2_並行Tool_Calling_DAG.ipynb` 學習如何最大化 Agent 的並行效率